# Hoffle DJ song tagger (CLAP)
Runtime → T4 GPU. Upload `clips.zip` (from `make_clips.py`) to Google Drive first.
Outputs `song_tags.json` + `embeddings.npy` — download them and hand to Claude.

In [ ]:
!pip -q install transformers librosa soundfile
from google.colab import drive
drive.mount('/content/drive')

Set the path to where you put `clips.zip` in Drive:

In [ ]:
CLIPS_ZIP = '/content/drive/MyDrive/clips.zip'
!rm -rf /content/clips && mkdir /content/clips && unzip -q -o "$CLIPS_ZIP" -d /content/clips
!ls /content/clips | wc -l

In [ ]:
import json, os, numpy as np, torch, librosa
from transformers import ClapModel, ClapProcessor

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device', dev)
model = ClapModel.from_pretrained('laion/larger_clap_music').to(dev).eval()
proc = ClapProcessor.from_pretrained('laion/larger_clap_music')

LABELS = {
 'genre': ['turkish pop','turkish rock','turkish rap','turkish alternative rock','arabesque turkish music','indie rock','alternative rock','garage rock','punk rock','pop punk','emo','hard rock','classic rock','heavy metal','nu metal','post-hardcore','grunge','indie pop','dream pop','synth pop','electropop','dance pop','house music','techno','drum and bass','hip hop','trap','r&b','soul','funk','disco','reggae','latin pop','k-pop','jazz','lofi','acoustic singer-songwriter','folk','country','classical','film score','ambient'],
 'mood': ['energetic and aggressive','upbeat and happy','chill and relaxed','sad and melancholic','romantic','dark and moody','dreamy','nostalgic','epic and anthemic','party music','angry','hopeful'],
 'vocal': ['male vocals','female vocals','rap vocals','instrumental with no vocals','group vocals'],
 'sound': ['distorted electric guitars','clean guitar','piano driven','synthesizers','heavy bass','acoustic guitar','orchestral strings','electronic drums','live drums'],
}
def text_emb(prompts, prefix):
    t = proc(text=[prefix + p for p in prompts], return_tensors='pt', padding=True).to(dev)
    with torch.no_grad():
        e = model.get_text_features(**t)
    return torch.nn.functional.normalize(e, dim=-1)
T = {k: text_emb(v, 'this is a song of the genre ' if k=='genre' else 'this music is ') for k, v in LABELS.items()}

ids = sorted(f[:-4] for f in os.listdir('/content/clips') if f.endswith('.mp3'))
embs, tags = [], {}
for i, vid in enumerate(ids):
    y, _ = librosa.load(f'/content/clips/{vid}.mp3', sr=48000, mono=True)
    inp = proc(audios=y, sampling_rate=48000, return_tensors='pt').to(dev)
    with torch.no_grad():
        e = torch.nn.functional.normalize(model.get_audio_features(**inp), dim=-1)
    embs.append(e[0].cpu().numpy())
    entry = {}
    for k, te in T.items():
        p = torch.softmax(100 * e @ te.T, dim=-1)[0]
        top = p.topk(3)
        entry[k] = [[LABELS[k][j], round(float(s), 3)] for s, j in zip(top.values, top.indices)]
    tags[vid] = entry
    if i % 100 == 0: print(i, len(ids), vid, entry['genre'][0], entry['mood'][0])
np.save('embeddings.npy', np.stack(embs))
json.dump({'ids': ids, 'tags': tags}, open('song_tags.json', 'w'), ensure_ascii=False)
print('done', len(ids))

In [ ]:
from google.colab import files
files.download('song_tags.json'); files.download('embeddings.npy')